# Ingesta de relaciones película-compañía (incremental)

Lee la carpeta `movie_company` (CSV) del lote `incremental_append/<p_file_date>` en Bronze, aplica un esquema explícito, estandariza las columnas y agrega `file_date`. Carga la tabla Delta `movie_silver.movies_companies` con `MERGE` (llave `movie_id` + `company_id` + `file_date`), particionada por `file_date`.

### Configuración

Carga las rutas y las funciones comunes, y lee los parámetros `p_environment` y `p_file_date` (fecha del lote).

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del folder movie_company

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, lit, current_timestamp

### Lectura del archivo

Primero lee la carpeta `movie_company` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
genre_df = (
    spark.read
        .format("csv")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie_company")
)

In [0]:
movies_company_df_shema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("companyId", StringType(), True)
])
movies_company_df = (
    spark.read
        .format('csv')
        .schema(movies_company_df_shema)
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie_company")
)

### Transformación de columnas

Renombra `movieId` y `companyId` a snake_case y agrega `ingestion_date`, `env` y `file_date`.

In [0]:
movies_company_df = (
    add_ingestion_date(movies_company_df)
    .select(
        col("movieId").alias("movie_id"),
        col("companyId").alias("company_id"),
        lit(v_env).alias("env"),
        lit(v_file_date).alias("file_date")
    )
)

### Escritura en Silver

Las celdas comentadas conservan estrategias anteriores (borrar partición + `append`). La carga activa es `merge_delta_lake` sobre `movie_silver.movies_companies` con la llave `movie_id` + `company_id` + `file_date`.

In [0]:
#overwrite_partitioned_table("movie_silver", "movies_companies", "file_date", v_file_date)

In [0]:
"""
(
    movies_company_df
    .write
        .mode("append")
        .partitionBy("file_date")
        .saveAsTable("movie_silver.movies_companies")
)
"""

In [0]:
merge_condition = 'trgt.movie_id = src.movie_id AND trgt.company_id = src.company_id AND trgt.file_date = src.file_date'
merge_delta_lake(input_df = movies_company_df, schema_name = "movie_silver", table_name = "movies_companies", merge_condition = merge_condition, partition_column = "file_date")

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")